In [1]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece huggingface_hub google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 91.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 31.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2

In [2]:
import gc
import torch

from kaggle_secrets import UserSecretsClient
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

In [3]:
secrets = UserSecretsClient()

try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Не удалось получить HF_TOKEN. "
        "Добавьте токен в Kaggle Secrets и разрешите доступ ноутбуку."
    ) from e


_loaded_models = {}

# Для ограниченной GPU-памяти
USE_4BIT = True

quantization_config = None

if torch.cuda.is_available() and USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )


# Только этим моделям передаём токен явно.
PRIVATE_MODELS = {
    "meta-llama/Llama-3.1-8B-Instruct",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.3-70B-Instruct",
    "meta-llama/Llama-3.2-1B"
}


def get_hf_auth_kwargs(model_id: str) -> dict:
    # Возвращает параметры авторизации для Hugging Face

    if model_id in PRIVATE_MODELS:
        return {"token": HF_TOKEN}

    return {}


def load_local_model(model_id: str):
    # Загрузка и сохранение модели в кэш

    if model_id in _loaded_models:
        return _loaded_models[model_id]

    auth_kwargs = get_hf_auth_kwargs(model_id)

    common_kwargs = {
        "device_map": "auto",
        "torch_dtype": (
            torch.float16
            if torch.cuda.is_available()
            else torch.float32
        ),
        **auth_kwargs,
    }

    if quantization_config is not None:
        common_kwargs["quantization_config"] = quantization_config

    if model_id == "google/gemma-3-4b-it":
        processor = AutoProcessor.from_pretrained(
            model_id,
            **auth_kwargs,
        )

        model = AutoModelForImageTextToText.from_pretrained(
            model_id,
            **common_kwargs,
        )
        model.eval()

        _loaded_models[model_id] = ("gemma", processor, model)

    else:
        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            **auth_kwargs,
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            **common_kwargs,
        )
        model.eval()

        _loaded_models[model_id] = ("causal", tokenizer, model)

    return _loaded_models[model_id]


def generate_fn(
    model_id: str,
    prompt: str,
    max_new_tokens: int = 512,
) -> str:
    # Генерирация ответа

    model_type, tokenizer_or_processor, model = load_local_model(
        model_id
    )

    messages = [{"role": "user", "content": prompt}]

    with torch.inference_mode():

        if model_type == "gemma":
            processor = tokenizer_or_processor

            formatted = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

            inputs = processor(
                text=[formatted],
                return_tensors="pt",
            ).to(model.device)

            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
            )

            input_length = inputs["input_ids"].shape[1]
            new_tokens = output[:, input_length:]

            result = processor.batch_decode(
                new_tokens,
                skip_special_tokens=True,
            )[0]

        else:
            tokenizer = tokenizer_or_processor

            formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

            inputs = tokenizer(
                formatted,
                return_tensors="pt",
            ).to(model.device)

            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

            input_length = inputs["input_ids"].shape[1]
            new_tokens = output[0, input_length:]

            result = tokenizer.decode(
                new_tokens,
                skip_special_tokens=True,
            )

    return result.strip()

In [8]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [6]:
test_prompt = "Briefly explain why the seasons change."

answer = generate_fn(
    "microsoft/Phi-3.5-mini-instruct",
    test_prompt,
    max_new_tokens=128,
)

print("\nmicrosoft/Phi-3.5-mini-instruct:\n",answer)

config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]


microsoft/Phi-3.5-mini-instruct:
 The change of seasons is primarily due to the tilt of the Earth's axis in relation to its orbit around the sun. The Earth's axis is tilted at an angle of approximately 23.5 degrees relative to its orbit. This tilt causes different parts of the Earth to receive varying amounts of sunlight throughout the year, leading to the changing seasons.

During the summer months, the hemisphere tilted towards the sun experiences more direct sunlight and longer days, resulting in warmer temperatures. Conversely, during the winter months, the hemisphere tilted away


In [9]:
test_prompt = "Briefly explain why the seasons change."

answer = generate_fn(
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    test_prompt,
    max_new_tokens=128,
)

print("\nHuggingFaceTB/SmolLM2-1.7B-Instruct:\n",answer)

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


HuggingFaceTB/SmolLM2-1.7B-Instruct:
 The seasons change due to the Earth's tilt in its orbit around the Sun. The Earth's axis is tilted at an angle of about 23.5 degrees, which means that the amount of sunlight a particular region receives varies throughout the year.

During the summer months, the Northern Hemisphere is tilted towards the Sun, resulting in longer days and more direct sunlight. This leads to warmer temperatures and longer days.

In the winter months, the Northern Hemisphere is tilted away from the Sun, resulting in shorter days and less direct sunlight. This leads to colder temperatures and shorter days.

The same process occurs in the Southern


In [4]:
test_prompt = "Briefly explain why the seasons change."

answer = generate_fn(
    "Qwen/Qwen2.5-7B-Instruct",
    test_prompt,
    max_new_tokens=128,
)

print("\nQwen/Qwen2.5-7B-Instruct:\n",answer)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


Qwen/Qwen2.5-7B-Instruct:
 The seasons change due to the tilt of the Earth's rotational axis and its revolution around the Sun. The Earth rotates on its axis at an angle of approximately 23.5 degrees relative to its orbital plane around the Sun. As the Earth orbits the Sun, this tilt causes different parts of the Earth to receive varying amounts of direct sunlight throughout the year.

During the summer in the Northern Hemisphere, for example, the North Pole is tilted towards the Sun, resulting in more direct sunlight and longer days, leading to warmer temperatures. Conversely, during winter in the Northern Hemisphere, the North Pole is tilted away from the Sun, leading to less


In [7]:
test_prompt = "Briefly explain why the seasons change."

answer = generate_fn(
    "mistralai/Mistral-Nemo-Instruct-2407",
    test_prompt,
    max_new_tokens=128,
)

print("\nmistralai/Mistral-Nemo-Instruct-2407:\n",answer)

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[transformers] The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


mistralai/Mistral-Nemo-Instruct-2407:
 The seasons change due to Earth's axial tilt and its revolution around the Sun. Here's a brief explanation:

1. **Axial Tilt**: Earth is tilted at an angle of about 23.5 degrees on its axis. This tilt causes the Sun's rays to hit Earth at different angles at different times of the year.

2. **Revolution Around the Sun**: Earth orbits the Sun once every 365.24 days. As it does so, different parts of Earth receive more or less sunlight, causing the seasons to change.

Here's how it works:

- **Summer (June, July, August


In [ ]:
from huggingface_hub import login
login()

In [16]:
test_prompt = "Briefly explain why the seasons change."

answer = generate_fn(
    "meta-llama/Llama-3.2-3B-Instruct",
    test_prompt,
    max_new_tokens=128,
)

print(answer)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The seasons change due to the Earth's tilt on its axis and its orbit around the Sun. Here's a simplified explanation:

1. **Earth's Tilt**: The Earth is tilted on its axis at an angle of about 23.5 degrees. This means that, as it orbits the Sun, different parts of the planet receive varying amounts of sunlight throughout the year.
2. **Orbit around the Sun**: The Earth takes approximately 365.25 days to complete one orbit around the Sun. As it moves, the tilt of its axis causes different parts of the planet to be tilted towards or away from the Sun.
3.


In [10]:
test_prompt = "Briefly explain why the seasons change."

answer = generate_fn(
    "google/gemma-2-9b-it",
    test_prompt,
    max_new_tokens=128,
)

print("\ngoogle/gemma-2-9b-it:\n",answer)

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]


google/gemma-2-9b-it:
 The Earth's tilt on its axis is the primary reason for the seasons. 

As the Earth orbits the Sun, different parts of the planet receive varying amounts of direct sunlight throughout the year. 

* **Summer:** When a hemisphere is tilted towards the Sun, it receives more direct sunlight, leading to warmer temperatures and longer days.
* **Winter:** When a hemisphere is tilted away from the Sun, it receives less direct sunlight, resulting in colder temperatures and shorter days.

This tilt remains constant throughout the year, causing the cycle of seasons.
